# 2.1 Who the sentence is about

Most sentences that carry sentiment about a company never name it. Journalists write "The company
also raised prices" and "It expects deliveries to keep climbing", and those sentences are a quarter
of the target population, which is far too much signal to throw away.

This notebook is the whole arc: the model that reads the mention chain, the two bugs we found by
reading its output, the text rewrite it makes possible, three attempts at auditing it, and the
judge that finally repaired it. We resolve, we rewrite, we audit, we discover the audit covered
only half the channel, we measure both halves, and we verify each sentence, which takes referent
error from 4.20% to 0.45%.

We relabelled the eval set under two convention changes on 2026-08-18, described in section 9, and
re-ran every cell here against it, so all the figures below are current.

In [1]:
import numpy as np
import pandas as pd
from stock_predictor.config import DATA_DIR, PROJ_ROOT

pd.set_option("display.max_colwidth", 140)
sentences = pd.read_parquet(DATA_DIR / "sentences.parquet")
body = sentences[~sentences["is_boilerplate"]]
print("non-boilerplate sentences:", len(body))
print("target sentences:", int(body["mentions_target"].sum()))

2026-08-16 22:03:31.234 | INFO     | stock_predictor.config:<module>:12 - PROJ_ROOT path is: D:\ML\stock-predictor


non-boilerplate sentences: 57250
target sentences: 13669


### 1. Coreference

Which company a referring expression points at is a discourse question, and two cases matter that
no rule over word order can express. The first is a multi-company roundup, where the nearest named
company is incidental rather than the topic: "Lucid's market debut was led by Peter Rawlinson,
Tesla's former chief vehicle engineer" hijacks every "it" in the Lucid paragraph that follows. The
second is attribution buried in a subordinate clause, where the company that is grammatically near
is not the one the pronoun refers back to.

`coref.py` wraps fastcoref (`biu-nlp/f-coref`, around 90M parameters, CPU by design). Three
decisions about how we wired it in matter more than the choice of model:

- **Degradation.** If the backend is missing or inference raises, we log one warning and tag on
  explicit names alone. Coreference never hard-fails the run.
- **Its own column.** `resolved_by_coref` records exactly which rows the model spoke for, which
  keeps them separately measurable from the rows that named the company outright. Section 4 is why
  that matters.
- **Character offsets rather than token indices.** The spans address the exact cleaned string the
  sentence Spans were parsed from, and `process_articles` asserts that identity rather than
  assuming it. Section 2 overwrites those characters.

Coreference is the most expensive stage in the text layer, so we cache clusters on a hash of the
document text. Cold it costs 833s and warm 0.2s, and we asserted the cold and warm tag tables are
byte-identical.

In [5]:
print("coref backend available:", coref.is_available())
coref_cache = coref.load_cache()
print("coref cache entries:", len(coref_cache))

D:\ML\stock-predictor\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


08/16/2026 11:01:20 - INFO - 	 HTTP Request: HEAD https://huggingface.co/biu-nlp/f-coref/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"


08/16/2026 11:01:20 - WARNING - 	 Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


08/16/2026 11:01:20 - INFO - 	 HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/biu-nlp/f-coref/e91dfff12879495d882ba9460d0c5d5dd44ade59/config.json "HTTP/1.1 200 OK"


2026-08-16 11:01:20.171 | INFO     | stock_predictor.text.coref:_load_model:114 - Loading coreference model 'biu-nlp/f-coref' (CPU)


...

08/16/2026 11:01:21 - INFO - 	 HTTP Request: GET https://huggingface.co/api/models/biu-nlp/f-coref/commits/main "HTTP/1.1 200 OK"


08/16/2026 11:01:21 - INFO - 	 HTTP Request: GET https://huggingface.co/api/models/biu-nlp/f-coref/discussions?p=0 "HTTP/1.1 200 OK"


2026-08-16 11:01:21.458 | INFO     | stock_predictor.text.coref:_load_model:116 - Coreference model loaded
coref backend available: True


08/16/2026 11:01:21 - INFO - 	 HTTP Request: GET https://huggingface.co/api/models/biu-nlp/f-coref/commits/refs%2Fpr%2F1 "HTTP/1.1 200 OK"


coref cache entries: 2073


#### 1.1 Two person-chain bugs

We found both the same way, by sampling the sentences where coreference and the older heuristic
disagreed and reading them. Neither would have surfaced from aggregate statistics.

The mention "Tesla CEO Elon Musk" contains "Tesla", so a plain substring test keyed an entire
person chain to the company. Article 140122179 is a story about Musk's lawsuit against Sam Altman,
nothing to do with Tesla, and the sentence "Kalshi's similar market has Musk winning at 55%." was
tagged as a Tesla sentence purely because the Musk chain contained that appositive. That bypassed a
rule which predates coreference entirely, since `mentions_target` has never been settable by a
person-tier match, as described in 2.0 section 3. Coreference had found a back door into it.

The residual case carries no PERSON entity at all. "the Tesla CEO" and "the Tesla billionaire's"
are appositive title phrases that name a person by their company. We fixed it with a job-title
token rule, extended after article 139905684 to person-denoting head nouns of the same
construction, such as "billionaire" and "tycoon".

Both fixes affect keying only. A person-like mention still belongs to its cluster and still tags
its sentence if some other mention resolves the chain. Across the corpus the pair removes around
174 `mentions_target` sentences.

This is also why spaCy's `ner` component stays enabled after we removed the ORG detector. PERSON
entities are load-bearing in three places, and turning `ner` off would break all three without
raising anything.

### 2. The rewrite

FinBERT scores a sentence, while the pipeline asks how a sentence reads about the target.
"Build-A-Bear outperformed Tesla" is a positive sentence and negative news for Tesla.

ABSA (`yangheng/deberta-v3-base-absa-v1.1`) is handed the aspect explicitly, but it attends to an
aspect term that has to be present in the text, since it is not a classifier over an entity id. A
sentence that only refers back contains nothing for it to attend to, so we inject the resolved name
over the mention's own characters, using the spans from section 1:

    "The company also raised prices."  ->  "Tesla also raised prices."

This makes ABSA quality depend on referent quality. A wrong referent no longer merely mis-files a
correct score, it changes the sentence the model reads, and the model then scores confidently
toward the company we injected. That dependency is why we verify coreference in sections 7 and 8
rather than trusting it.

The comparison of ABSA against FinBERT as scorers is notebook 2.2. Only the rewrite belongs here,
because the rewrite is a consequence of coreference.

In [6]:
# The pre-2.3 snapshot is the unaudited state: every row where the injected text
# differs from the original sentence.
sub_before = sentences_before[
    (sentences_before["absa_text"] != "")
    & (sentences_before["absa_text"] != sentences_before["text"])
].copy()

sub_before["surface"] = sub_before.apply(
    lambda r: r["text"][int(r["mention_char_start"]): int(r["mention_char_end"])], axis=1
)
print(f"substitutions performed: {len(sub_before)}")
print()
print("what was replaced, most common 12:")
print(sub_before["surface"].str.lower().value_counts().head(12).to_string())

substitutions performed: 2845

what was replaced, most common 12:
surface
the company      630
it               418
its              241
the company's    226
we               207
the stock        157
they             102
the company’s     46
our               44
the ev maker      34
the automaker     24
the firm          20


#### 2.1 The fifth defect class

An earlier pass found and fixed four defects: an expletive "it" being substituted, possessives
producing broken grammar, person mentions being overwritten, and NER-sourced companies with no
usable aspect. All four are genuinely fixed, they have tests, and the corpus numbers reproduce.

Reading a fresh sample turned up a fifth class that none of those four addresses, because each of
them patched one named failure and nothing constrained the injected span to be a company-referring
expression in general.

In [7]:
# The defect: spans that are not company mentions at all.
defects = [
    "a school bus with its stop arm extended and red lights flashing",
    "he",
    "brazil",
    "both stocks",
    "'s",
    "the earnings call",
]
show = sub_before[sub_before["surface"].str.lower().isin(defects)].drop_duplicates("surface")
for _, r in show.iterrows():
    print(f"replaced {r['surface']!r}:")
    print(f"  ORIG: {r['text'][:150]}")
    print(f"  ABSA: {r['absa_text'][:150]}")
    print()

replaced 'He':
  ORIG: He he led the finance team there.
  ABSA: Tesla he led the finance team there.

replaced 'the earnings call':
  ORIG: It was a remarkable moment in the earnings call.
  ABSA: It was a remarkable moment in Tesla.

replaced 'he':
  ORIG: And the cost of home charging is one tenth of petrol prices," he told reporters ⁠in
  ABSA: And the cost of home charging is one tenth of petrol prices," Tesla told reporters ⁠in

replaced "'s":
  ORIG: Let's get what we can and the $99 price point is an easier sell, I think.
  ABSA: LetTesla's get what we can and the $99 price point is an easier sell, I think.

replaced 'The earnings call':
  ORIG: The earnings call came a few days after the company changed its mission from accelerating the “world's transition to sustainable energy” to building “
  ABSA: Tesla came a few days after the company changed its mission from accelerating the “world's transition to sustainable energy” to building “a world of a

replaced 'a school bus with

Six shapes of the same mistake, all of them coreference errors or non-substitutable mention forms
that the substitution then carried out faithfully:

- a product description: "...whether FSD would stop for **Tesla**."
- a person pronoun: "**Tesla** he led the finance team there."
- another proper noun: "That list is Australia, **Tesla**, India, Korea..."
- a plural: "**Tesla** trade at steep valuations"
- a bare clitic: "Let**Tesla's** get what we can"
- an arbitrary noun phrase: "It was a remarkable moment in **Tesla**."

#### 2.2 The guard

`is_substitutable_mention` allows two shapes. The first is a closed set of non-person pronouns,
`it`, `its`, `they` and `their`. The second is a short determiner-headed noun phrase whose head
denotes a company or its stock, such as "the company", "this electric car maker" or "the
automaker's". Everything else is refused.

Person pronouns are refused by being absent from the allowed set, which keeps the existing person
rule and this one saying the same thing.

We removed first-person plural on measured evidence. In news text `we`, `our` and `us` are nearly
always inside a quote from a person, and "'We want the future to look like the future,' Musk said"
became "Tesla want the future to look like the future". There were ten such cases.

In [8]:
checks = [
    "it", "its", "the company", "the company's", "this electric car maker", "company",
    "he", "she", "his company", "Brazil", "Spotify", "robotaxi", "the Model Y",
    "both companies", "the shares", "'s", "that", "the earnings call",
]
pd.DataFrame(
    {"surface": checks,
     "substitutable": [entity_filter.is_substitutable_mention(c) for c in checks]}
)

,surface,substitutable
0,it,True
1,its,True
2,the company,True
3,the company's,True
4,this electric car maker,True
5,company,True
6,he,False
7,she,False
8,his company,False
9,Brazil,False


Applying the guard when the span is chosen, rather than filtering after the fact, is measurably
better. 442 of the 2,845 substitutions were over a non-referring span, but the number of
substitutions falls by only 380, because for 62 sentences the guard causes a different and valid
mention in the same sentence to be chosen instead.

"The EV maker's shares" is the clearest of them. The whole phrase used to be overwritten, giving
"Tesla have been hot lately, rising about 32% this month", and the span now lands on the company
part alone, so it reads "Tesla's shares have been hot lately".

A refusal is not a lost row either. We score the sentence with its text unchanged, which is the
same position FinBERT is already in, rather than with a confidently wrong injection.

We apply the guard twice, once at span selection and again in `absa._substitute_resolved`, so a bad
span cannot get an injection past it. No non-referring substitutions remain in the corpus.

The cell below lost its stored output when we renamed the span columns.

In [ ]:
def _substituted(df):
    m = (df["absa_text"] != "") & (df["absa_text"] != df["text"])
    out = df[m].copy()
    out["surface"] = out.apply(
        lambda r: r["text"][int(r["mention_char_start"]):int(r["mention_char_end"])], axis=1
    )
    return out.set_index(["article_id", "sent_idx"])

b, a = _substituted(sentences_before), _substituted(scored_sentences)
b["ok"] = b["surface"].map(entity_filter.is_substitutable_mention)
bad = b[~b["ok"]]
repointed = bad.index.intersection(a.index)

print(f"spans that were not company mentions : {len(bad)}")
print(f"substitutions refused outright       : {len(b) - len(a)}")
print(f"re-pointed to a valid mention        : {len(repointed)}")
print()
for key in list(repointed)[:5]:
    print(f"  was replacing {bad.loc[key, 'surface']!r}")
    print(f"    BEFORE: {b.loc[key, 'absa_text'][:120]}")
    print(f"    AFTER : {a.loc[key, 'absa_text'][:120]}")

# The closing check: nothing non-referring survives anywhere in the new table.
a["ok"] = a["surface"].map(entity_filter.is_substitutable_mention)
print()
print(f"non-referring substitutions remaining: {int((~a['ok']).sum())}")

### 3. Auditing the referent

It took three attempts, and the two failures are worth more than the method that worked.

**Attempt one was worthless.** We showed the auditor each sentence on its own and asked whether it
was about Tesla, and it came back at 0.59, which we nearly reported as precision. It is not
precision. Coreference resolves pronouns using article context, and we had removed the context. For
exactly the sentences the mechanism exists to handle, such as "It's a wonderful business", the
auditor could not possibly verify the referent, and our instructions said to answer `no` when no
company could be identified. So "the algorithm was wrong" and "this sheet cannot tell" collapsed
into the same number, and roughly half the failures were the second kind. The auditor flagged it
itself, saying the reported `no` rate overstates the true error rate and should be read as
"unverifiable from sentence alone". An evaluation has to give the judge at least the evidence the
system had.

**Attempt two fixed that and broke something else.** We added context of 5 preceding and 1
following sentence, asked for the referent as free text rather than asking whether our claim was
right, and made `AMBIGUOUS` a first-class verdict, since a confident tag is unjustified either way
if a careful reader with context still cannot tell. It gave coreference 74%, determinate in 99% of
rows. But our mask replaced "Tesla's" with `<NAME>`, which ate the possessive, so a correct
substitution was shown to the auditor as a dropped possessive and duly marked broken. 40 of the 85
reported substitution failures were that artifact. The tell was in the auditor's own report, which
noted there was not a single `<NAME>'s` anywhere in the sheet, and that should have been impossible.

**Attempt three is the clean one.** 200 coreference-resolved sentences, context intact, referent
asked first, and the mask now mapping "Tesla's" to `<NAME>'s` so the grammar survives and only the
identity is hidden. We asserted the sheet contained `<NAME>'s` rows before sending it.

The sheet and its key, `references/context-audit-sheet-v2.csv` and
`context-audit-key-v2.parquet`, were deleted on 2026-08-21 once they had been consumed. What we
kept is the part that could not be regenerated, `references/context-audit-labels-v2.csv`, and every
`sample_id` in it encodes the article and sentence index, so the audited rows rejoin to the sentence
table without the sheet. The cells below that read the sheet no longer run, and their stored output
is the record.

In [3]:
key = pd.read_parquet(PROJ_ROOT / "references" / "context-audit-key-v2.parquet") \
    if (PROJ_ROOT / "references" / "context-audit-key-v2.parquet").exists() else None
lab = pd.read_csv(PROJ_ROOT / "references" / "context-audit-labels-v2.csv")
sheet = pd.read_csv(PROJ_ROOT / "references" / "context-audit-sheet-v2.csv")
d = sheet[["sample_id", "highlighted_phrase"]].merge(lab, on="sample_id")
# pandas reads TRUE/FALSE as booleans and NA as NaN, so normalise via str.
d["ref"] = d["refers_to_tesla"].astype(str).str.upper()
d["span"] = d["substitution_span_ok"].astype(str).str.upper()
d["has_decision"] = d["highlighted_phrase"].notna() & (d["highlighted_phrase"].astype(str) != "")

print(f"rows audited: {len(d)}")
print(f"  carrying a resolution decision: {int(d['has_decision'].sum())}")
print(f"  no phrase resolved            : {int((~d['has_decision']).sum())}")

rows audited: 200
  carrying a resolution decision: 138
  no phrase resolved            : 62


The auditor caught a sampling flaw. 62 of the 200 rows carry no highlighted phrase, because they
are coreference-tagged but nothing substitutable was found, so there is no resolution decision to
judge. We had sampled on the tag rather than on the presence of a span. Those rows are excluded
below, since counting them as errors repeats attempt one exactly.

In [4]:
real = d[d["has_decision"]]
print("=== REFERENT (rows carrying a decision) ===")
print(real["ref"].value_counts().to_string())
print(f"precision: {(real['ref'] == 'TRUE').mean():.3f}")
print()
sub = real[real["span"].isin(["TRUE", "FALSE"])]
print(f"=== SUBSTITUTION: {(sub['span'] == 'TRUE').mean():.3f} sound  (n={len(sub)}) ===")
print()
bad = real[(real["ref"] != "TRUE") | (real["span"] == "FALSE")]
print("failure classes:")
print(bad["note"].astype(str).str.lower().value_counts().to_string())

=== REFERENT (rows carrying a decision) ===
ref
TRUE         129
FALSE          8
AMBIGUOUS      1
precision: 0.935

=== SUBSTITUTION: 0.957 sound  (n=138) ===

failure classes:
note
refers to a different company                3
refers to a non-company antecedent           3
plural referent - refers to two companies    2
subject-verb disagreement                    1
antecedent outside context window            1


That is 93.5% correct referent and 95.7% mechanically sound substitutions over 138 decisions. Five
of the six substitution failures are downstream of a referent error, and only one broke while the
referent was right.

The residual errors are a short list and none of them has a cheap fix. Three resolved to a
different company, three to a non-company antecedent, two collapsed a plural referent onto one
company, and one had its antecedent above the context window. The non-company case is worth
reading: in "Tesla's US market share dropped to 45%. In 2019, it was 80%", the antecedent is the
figure rather than the company.

The auditor found the passage determinate in 137 of 138 rows, so context depth is almost never the
binding constraint, and whatever headroom is left sits in the resolver.

On method, two of the three attempts were invalidated by our own design errors, and the auditor's
report caught both of them rather than we did. In both cases the tell was a number too extreme to
be plausible: 67% of everything wrong, and not one possessive in 213 substitutions. When an
evaluation says almost everything is broken, suspect the evaluation first.

### 4. That figure covers three quarters of the channel

The 93.5% is measured on rows where coreference produced a substitutable span. Around a third of
coreference-tagged sentences carry no usable span at all. They are still tagged as target
sentences and still scored, on unchanged text with no aspect anchor, and the audit says nothing
about whether those tags are right.

Some of that third exists because of the fixes in section 2.2. A sentence whose only mention was
"we" now correctly yields no span, which suppresses the bad substitution and leaves the tag in
place.

So we sampled the no-span population in its own right: 120 new rows, seed `20260818`, disjoint from
the original set, with the same context window every labelling pass in this branch uses, which is
the headline, 4 preceding sentences, the sentence itself, and 1 following sentence. Each row was
judged `target` or `other`, with a free-text referent and a `borderline` flag.

In [ ]:
import numpy as np
import pandas as pd

eval_df = pd.read_parquet("../../data/eval/coref_eval_labelled.parquet")
print(f"Total labelled rows: {len(eval_df)}")
print(eval_df.groupby("has_span").size().rename("n"))
eval_df.head(3)

In [2]:
def wilson_ci(k, n, z=1.96):
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return center - half, center + half


span = eval_df[eval_df["has_span"]]
nospan = eval_df[~eval_df["has_span"]]

span_k, span_n = int((span["verdict"] == "target").sum()), len(span)
nospan_k, nospan_n = int((nospan["verdict"] == "target").sum()), len(nospan)

rows = [
    ("span (rewritten)", span_k, span_n),
    ("no-span (tagged only) -- OLD n=50", 26, 50),
    ("no-span (tagged only) -- NEW n=170", nospan_k, nospan_n),
]
for label, k, n in rows:
    lo, hi = wilson_ci(k, n)
    print(f"{label:38s} {k:3d}/{n:3d} = {k/n:6.1%}   95% CI [{lo:.1%}, {hi:.1%}]")

span (rewritten)                        90/100 =  90.0%   95% CI [82.6%, 94.5%]
no-span (tagged only) -- OLD n=50       26/ 50 =  52.0%   95% CI [38.5%, 65.2%]
no-span (tagged only) -- NEW n=170     117/170 =  68.8%   95% CI [61.5%, 75.3%]


The two populations are 21 points apart. The point estimate moved from the original n=50 sample's
52.0% to 68.8%, and that shift is the convention changes in section 9 rather than sampling noise.
The interval narrowed from plus or minus 13.4pp to plus or minus 7.4pp.

Weighted by what each population actually contributes:

In [3]:
# Current population sizes, recomputed from the live corpus (data/sentences.parquet), not the
# handoff's earlier snapshot -- these happen to be unchanged this session (corpus not regenerated).
sentences = pd.read_parquet("../../data/sentences.parquet")
coref = sentences[sentences["resolved_by_coref"]]
span_pop = int(coref["mention_char_start"].notna().sum())
nospan_pop = int(coref["mention_char_start"].isna().sum())
target_pop = int((sentences["mentions_target"] & ~sentences["is_boilerplate"]).sum())

span_acc = span_k / span_n
nospan_acc = nospan_k / nospan_n
blended = (span_pop * span_acc + nospan_pop * nospan_acc) / (span_pop + nospan_pop)

print(f"coref-resolved sentences   : {len(coref):6d}")
print(f"  with span (rewritten)    : {span_pop:6d}  ({span_pop/len(coref):.1%})")
print(f"  no span (tagged only)    : {nospan_pop:6d}  ({nospan_pop/len(coref):.1%})")
print(f"coref share of target set  : {len(coref)/target_pop:.1%}")
print()
print(f"span accuracy              : {span_acc:.1%}")
print(f"no-span accuracy           : {nospan_acc:.1%}")
print(f"BLENDED (population-weighted): {blended:.1%}")

coref-resolved sentences   :   3617
  with span (rewritten)    :   2428  (67.1%)
  no span (tagged only)    :   1189  (32.9%)
coref share of target set  : 24.8%

span accuracy              : 90.0%
no-span accuracy           : 68.8%
BLENDED (population-weighted): 83.0%


The coreference channel runs at 83.0%, not at 93.5%, and 90.0% describes span rows only.

In absolute terms, no-span is 32.9% of the coreference population, and coreference is 24.8% of the
non-boilerplate target set, so roughly 1 target sentence in 12 arrives through a channel that is
right about two thirds of the time. Two populations 21 points apart cannot be averaged into one
number and handed to a model as though the sentences behind them were equally trustworthy. Given
the split a model can discount one of them, and given only the blend it cannot, because the
information was destroyed before it arrived. Notebook 2.3 section 7 carries the split.

### 5. Each channel measured on its own

The 270 rows in section 4 split coreference by span and say nothing about `surface`, which is
roughly three quarters of the target population and had never been audited at all. The reasoning
for leaving it alone was that a sentence naming the company literally cannot be about anyone else.
So we sampled each channel separately, 150 rows apiece, using the same context window and the same
`target` or `other` question with a free-text referent and a `borderline` flag.

- **`coref_span`**: 146 of 150 correct, 97.3%, with 16 rows flagged borderline.
- **`coref_nospan`**: 139 of 150 correct, 92.7%, with 24 borderline.
- **`surface`**: 124 of 150 correct, 82.7%, with 31 borderline.

Surface is the weakest of the three, which inverts the assumption the pipeline had been running on.
Its failures are the cases the reasoning did not anticipate: funds and baskets that hold the target,
index roundups where the company is one name in a list, and sentences that name the company only
inside a comparison drawn about someone else.

These figures do not reconcile with section 4's 90.0% and 68.8%, and we have not established why.
The two rounds share 25 sentences and no row identifiers, and their referent conventions differ in
the sheets themselves, so the most likely explanation is that they were labelled under the two
conventions recorded in section 9 rather than that anything in the pipeline moved between them.
Until that is resolved, section 4 is the figure the judge is measured against, because it is the
set the harness actually loads.

Whether a wrong referent damages the score is a different question, since a sentence can be tagged
to the wrong company and still carry a score near zero. We sampled the same three channels again,
150 rows apiece, and labelled the outcome rather than the referent. `surface` was drawn twice as an
internal consistency check.

| channel | n | correct | benign | minor | harmful |
|---|---:|---:|---:|---:|---:|
| `coref_span` | 150 | 96.7% | 1.3% | 1.3% | 0.7% |
| `coref_nospan` | 150 | 74.0% | 23.3% | 2.0% | 0.7% |
| `surface` (a) | 150 | 60.0% | 32.0% | 4.0% | 4.0% |
| `surface` (b) | 150 | 70.7% | 18.7% | 2.0% | 8.7% |

`benign` means the referent was wrong and the score was close enough to zero that nothing reached
the article. Reading the two together changes which channel we worry about: `coref_nospan` is the
weakest on referent among the coref rows, but a quarter of its errors are benign and its harmful
rate is 0.7%, while `surface` carries the highest harmful rate at 4.0% and 8.7% across two draws.
The channel that is wrong most often is not the channel that does the most damage.

The two surface draws disagree by 10.7 points on `correct` and by 4.7 on `harmful`, which is the
honest measure of how much precision 150 rows buys: roughly none at this resolution. Every figure
in this section should be read as an indication rather than a measurement.

The labelling for both rounds was done by a model, not by a person. That is a real limit on all of
it, and it matters most in section 8, where an LLM judge is scored against labels an LLM produced.

### 6. The error tail

The errors are not a handful of confusable companies. We count them here before deciding what could catch them.


In [4]:
errors = eval_df[eval_df["verdict"] == "other"]
nospan_errors = errors[~errors["has_span"]]

print(f"Total errors across the full 270-row set : {len(errors)}")
print(f"  distinct referents                      : {errors['referent'].nunique()}")
print(f"No-span errors                            : {len(nospan_errors)}")
print(f"  distinct referents                      : {nospan_errors['referent'].nunique()}")
print()
errors["referent"].value_counts().head(12)

Total errors across the full 270-row set : 63
  distinct referents                      : 46
No-span errors                            : 53
  distinct referents                      : 43



referent
SpaceX                                7
BYD                                   5
Rivian                                3
Nova (Kimbal Musk's drone company)    3
Slate Auto                            2
SpaceX and xAI                        2
WeRide                                1
Unitree                               1
OpenAI                                1
Chancellor McCormick / LinkedIn       1
Agtonomy / DBL Partners               1
Uber                                  1
Name: count, dtype: int64

There are 46 distinct referents across 63 errors, and 43 distinct across the 53 no-span errors. The
head is short, with SpaceX at 7, BYD at 5, Rivian at 3 and Nova, which is Kimbal Musk's drone
company, at 3. After that comes a long tail of one-offs: Slate Auto, WeRide, Unitree, OpenAI,
Agtonomy, Uber, and a Delaware court filing.

That is the argument against a curated roster of confusable companies. No list of a practical size
reaches the tail, and under the ticker-agnostic constraint the tail looks entirely different for
another target. Asking what a sentence is about, rather than enumerating what it might be confused
with, is what the data supports, and it is the framing section 8 adopts.

The wider sample also surfaced failure shapes that were not visible at n=50:

- **Fund and ETF articles**: sentences from an article about a fund that merely holds the target
  resolve to the fund.
- **Musk-family companies**: a first-person "we" is a strong Tesla signal in an earnings call and an
  actively misleading one in an interview about a sibling venture.
- **Multi-topic headlines**: "Tesla European sales, Lucid Q4 earnings, Lamborghini: EV latest"
  produces sentences entirely about the other company that still carry a Tesla tag.
- **Generic technical or legal definitions**: a sentence stating the SAE Level 2 standard rather
  than describing Tesla.
- **Splitter fragments**: a bare "Li Auto Inc. (NASDAQ: LI)" that resolved to Tesla. That one is an
  upstream bug, in sentence boundary detection.

We had two items recorded as open. Four no-span errors looked like promotional boilerplate that
`is_boilerplate` had missed, and `SPAN-65` had a recorded span but an empty `absa_text`.

In [5]:
empty_absa = eval_df[eval_df["absa_text"].fillna("") == ""]
check = empty_absa.merge(
    sentences[["article_id", "sent_idx", "is_boilerplate"]],
    on=["article_id", "sent_idx"],
    how="left",
)
check[["row_id", "has_span", "is_boilerplate", "text"]]

,row_id,has_span,is_boilerplate,text
0,SPAN-65,True,True,The company has multiple vehicles in its fleet...
1,NOSPAN-31,False,True,Any views or opinions expressed may not reflec...
2,NOSPAN-38,False,True,Want the latest recommendations from Zacks Inv...
3,NOSPAN-44,False,True,Read the complete narrative.
4,NOSPAN-82,False,True,Contact Zacks Investment Research 800-767-3771...
5,NOSPAN-131,False,True,"Get stock recommendations, portfolio guidance,..."
6,NOSPAN-162,False,True,Get the latest stock analysis from Benzinga?


Both readings were wrong. `is_boilerplate` is already True for all seven rows including `SPAN-65`,
and `needs_score` correctly excludes them from both scorers, which is exactly why `absa_text` is
empty. `SPAN-65`'s text repeats across exactly 5 articles, which is the threshold. Both
mis-diagnoses came from judging boilerplate by eye from a sheet that did not carry the column, so
the lesson is to check the flag rather than infer it from the text.

### 7. Two stages that did not ship

Measuring the channel repaired nothing, so four stages followed, run lightest first so that each
stage's effect could be attributed to one change. Two of them shipped. We record the other two here
so nobody re-attempts them.

**Coreference ensemble disagreement (Maverick).** We ran a second, independently built backend over
the eval articles, to stand in for the per-link confidence fastcoref does not expose. Disagreement
is genuinely enriched for wrong resolutions, since 34 of the errors were active contradictions
rather than abstentions, and as a gate on span rows it halved residual error for a 3.4% cost.
Measured against the judge in section 8 it then bought 0.5pp for 122 fewer sentences, and we
withdrew the recommendation. A follow-on question, whether Maverick is simply the better backend,
we answered by labelling the 166 rows fastcoref never tagged: it is 3.9pp better, genuinely
different rather than merely more conservative, and still not worth swapping, because a swap
invalidates a corpus-wide cache in order to improve the channel that was already at 90%.
`COREF_MODEL` is unchanged.

**Two earlier suppression attempts**, which predate all of this, both asked an anomaly-detection
question, namely whether some other company is mentioned near the chain, and both shipped on
argument without being measured. Both discarded around 90% of the wrong rows they targeted, but
only by discarding most of the right ones as well. The one thing separating them from what follows
is that section 7 now exists first.

### 8. The eval harness

`coref_eval.py` changes no number in this notebook. Its job is to make a judge measurable before
the judge exists. It guarantees five properties, and each one maps to a way an earlier attempt went
wrong:

1. A judge is a callable with a row in `evaluate_judge`'s output, or it is not a judge.
2. The judge reads exactly what the human read. One shared `build_context` serves both the
   labelling sheets and the judges, and `verify_contexts_match` turns drift between corpus and
   labels into a loud failure rather than a plausible wrong number.
3. The headline metric is accept-precision rather than accuracy or F1. Of the rows a judge accepts,
   how many were genuinely about the target, since rejected rows never reach the sentiment model at
   all. A judge that discards half the corpus and is right about everything it keeps is a good
   outcome here, and a judge with good F1 and 85% accept-precision is not.
4. Accept-everything sits next to every judge, and nothing is reported pooled across span and
   no-span.
5. Verdicts are cached on `(article_id, sent_idx, target, model_id, prompt_version)`.
   `prompt_version` is in the key because without it the likeliest failure of the whole stage is
   quietly scoring a new prompt with the old prompt's answers, which looks like a working cache and
   produces a fabricated result. There is a test for it.

`unsure` is a discard rather than a third outcome. `accept_only` accepts on exactly `yes`, and a
judge that raises, times out or answers in a paragraph is discarded too. Losing a sentence is
acceptable and corrupting one is not.

In [9]:
# The harness against the real 270 rows: baseline and stub judges.
import sys

sys.path.insert(0, "../..")
from stock_predictor.text.coref_eval import (  # noqa: E402
    build_contexts,
    evaluate_judge,
    load_eval_set,
)

ev = load_eval_set()
arts = pd.read_parquet("../../data/articles.parquet")
ctxs = build_contexts(ev, sentences, arts)  # verify_contexts_match runs inside evaluate_judge

# An oracle judge: cheats by reading the label. Not a candidate -- it exists to
# prove the metrics compute correctly, and to show the ceiling any real judge
# is working toward.
truth = dict(zip(zip(ev["article_id"], ev["sent_idx"]), ev["verdict"]))
def oracle(ctx):
    return "yes" if truth[(ctx.article_id, ctx.sent_idx)] == "target" else "no"

report = pd.concat([
    evaluate_judge(lambda c: "yes", ev, contexts=ctxs, model_id="accept-all", use_cache=False),
    evaluate_judge(oracle, ev, contexts=ctxs, model_id="oracle (cheats)", use_cache=False),
])

show = ["judge", "population", "borderline", "n", "n_errors", "n_accepted",
        "accept_precision", "accept_precision_lo", "accept_precision_hi",
        "error_recall", "correct_lost"]
out = report[report["judge"] != "baseline (accept all)"][show]
print(out.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

print("\nBorderline rows are NOT a cosmetic slice:")
print(f"  total flagged borderline : {int(ev['borderline'].sum())} of {len(ev)}")
print(ev.groupby(["has_span", "borderline"]).size().to_string())

2026-08-18 12:45:18.964 | INFO     | stock_predictor.config:<module>:12 - PROJ_ROOT path is: D:\ML\stock-predictor


2026-08-18 12:45:18.980 | INFO     | stock_predictor.text.coref_eval:load_eval_set:231 - Loaded 270 labelled rows from D:\ML\stock-predictor\data\eval\coref_eval_labelled.parquet (100 span, 170 no-span, 63 errors)


2026-08-18 12:45:19.669 | INFO     | stock_predictor.text.coref_eval:evaluate_judge:502 - evaluate_judge: 270 rows (0 cache hits, 270 judged, 0 judge failures counted as 'unsure')
2026-08-18 12:45:19.687 | INFO     | stock_predictor.text.coref_eval:evaluate_judge:502 - evaluate_judge: 270 rows (0 cache hits, 270 judged, 0 judge failures counted as 'unsure')
          judge population borderline   n  n_errors  n_accepted  accept_precision  accept_precision_lo  accept_precision_hi  error_recall  correct_lost
     accept-all       span   included 100        10         100             0.900                0.826                0.945         0.000             0
     accept-all    no-span   included 170        53         170             0.688                0.615                0.753         0.000             0
     accept-all        all   included 270        63         270             0.767                0.713                0.813         0.000             0
     accept-all       span   exc

The oracle row is the ceiling, accept-all is the floor, and the gap between them is what a judge
competes for.

The harness immediately caught a stale constant. There are 37 borderline rows rather than 8, and
the 8 was true of the original 150-row set and never updated when we widened the sample. It matters
more than a corrected count usually would, because 36 of the 37 are no-span rows, so the two
borderline treatments are two materially different samples of the no-span channel rather than a
sensitivity check across the board.

The harness has limits we should not expect it to exceed. It measures a judge against 270 rows
drawn from fastcoref's output, so it says nothing about sentences fastcoref never tagged, and the
span channel holds only 10 errors, so span-side error recall carries a confidence interval around
plus or minus 25pp no matter how good the judge is. Accept-precision is the number this sample can
actually support.

### 9. The judge

The tail in section 5 is 46 distinct referents, so no curated roster reaches it, and a roster would
have to be rebuilt for every new target anyway. On that evidence we ruled out several options
before building any of them: sentence embeddings, because Tesla and SpaceX embed close together and
that is the wrong signal; extractive QA; NLI; supervised training on 270 labels; and KB entity
linking. What is left is a model that already knows, from pre-training, that Lightship is an RV
startup and BYD is a Chinese carmaker.

We use Qwen2.5-7B-Instruct at 4-bit (Q4_K_M GGUF) on CPU through `llama-cpp-python`, chosen over
transformers because it is independent of torch and cannot drag a different build in underneath
FinBERT, ABSA and fastcoref.

The gate runs per sentence at the rewrite site rather than per chain. A chain can be internally
incoherent, correct for some mentions and wrong for others, and no per-chain verdict can express
that.

A GBNF grammar admits exactly `yes`, `no` and `unsure`, so malformed output is impossible by
construction rather than something we clean up afterwards with a regex over free text. That removes
the class of bug where the model says "Yes, because..." and the parser takes the first word, and it
makes `unsure` a verdict the model chose rather than a bucket the parser fell into.

The context is what the labellers read, which measures at a median of 385 tokens and a maximum of
513 against `n_ctx = 4096`, so nothing is truncated. There are two prompts. Span rows are asked
about the marked phrase, which is the exact characters the pipeline will overwrite, and no-span
rows are asked about the marked sentence, which is the weaker claim the pipeline makes about them.

We wrote three prompt versions. v1 said "products or vehicles", which is an automaker assumption,
and instructed `no` for the target's own products, which contradicts the labels; it scored 100.0%
on span and 84.7% on no-span. v2 fixed those but broadened the accept criterion far past products
with "or its business, services or business units", and scored 94.9% and 83.3%. v3 accepts
products, carries no sector vocabulary and no vague broadening, and scores 98.4% and 78.1%.

v1 scores best and is still the wrong choice. It breaks the ticker-agnostic constraint, it
contradicts the labels, and its perfect span score partly depends on the model disobeying it: on
`SPAN-16` the prompt said products should be answered `no`, the model answered `yes`, and the label
says `yes`. The three intervals overlap heavily, so this sample cannot separate them, and where
measurement cannot decide we take the principled option rather than the best point estimate. v3
ships.

In [11]:
# Results, recomputed from the verdict cache under the current labels.
from stock_predictor.text.coref_eval import accept_only, load_judge_cache  # noqa: E402

cache = load_judge_cache()
cache = cache[cache["model_id"] == "qwen2.5-7b-instruct-q4km"]

ev = load_eval_set()  # reload: the labels changed since §9
d = ev.merge(cache[cache["prompt_version"] == "v3"][["article_id", "sent_idx", "answer"]],
             on=["article_id", "sent_idx"])
d["acc"] = d["answer"].map(accept_only)
d["tgt"] = d["verdict"] == "target"

print("Judge answers:", d["answer"].value_counts().to_dict(), "\n")

SPAN_POP, NOSPAN_POP = 2428, 1189
rates = {}
print(f"{'channel':16s}{'err before':>12s}{'err after':>11s}{'95% CI':>18s}{'kept':>9s}{'err recall':>12s}")
for label, sub in [("coref_span", d[d["has_span"]]), ("coref_nospan", d[~d["has_span"]])]:
    kept = sub[sub["acc"]]
    lo, hi = wilson_ci(int(kept["tgt"].sum()), len(kept))
    recall = ((~sub["acc"]) & (~sub["tgt"])).sum() / max((~sub["tgt"]).sum(), 1)
    rates[label] = {"keep": len(kept) / len(sub), "before": 1 - sub["tgt"].mean(),
                    "after": 1 - kept["tgt"].mean()}
    print(f"{label:16s}{1 - sub['tgt'].mean():11.1%}{1 - kept['tgt'].mean():11.1%}"
          f"   [{1-hi:5.1%},{1-lo:5.1%}]{len(kept)/len(sub):9.1%}{recall:12.1%}")

# Project onto the corpus populations (§3).
sent = sentences.copy()
sent["__channel"] = provenance_channel(sent)
target_pop = sent[sent["mentions_target"].fillna(False) & ~sent["is_boilerplate"].fillna(False)]
n_surface = int((target_pop["__channel"] == "surface").sum())

kept_n = {"coref_span": SPAN_POP * rates["coref_span"]["keep"],
          "coref_nospan": NOSPAN_POP * rates["coref_nospan"]["keep"]}
wrong_before = sum(p * rates[c]["before"] for c, p in [("coref_span", SPAN_POP),
                                                       ("coref_nospan", NOSPAN_POP)])
wrong_after = {c: kept_n[c] * rates[c]["after"] for c in kept_n}
total_before, total_after = len(target_pop), n_surface + sum(kept_n.values())
W = sum(wrong_after.values())

print(f"\nWHOLE TARGET-SENTENCE SET")
print(f"  before judge : {wrong_before:6.0f} wrong of {total_before:6d} = {wrong_before/total_before:5.2%}")
print(f"  after  judge : {W:6.0f} wrong of {total_after:6.0f} = {W/total_after:5.2%}"
      f"   (keeps {total_after/total_before:.1%} of sentences)")
print("\n  remaining measured error:")
for c in wrong_after:
    print(f"    {c:14s}{wrong_after[c]:6.0f}  ({wrong_after[c]/W:5.1%})")
print(f"\n  surface is UNMEASURED: {n_surface:,} rows = {n_surface/total_after:.0%} of what survives")
for e in [0.005, 0.01, 0.02]:
    sw = n_surface * e
    print(f"    if surface error = {e:4.1%} -> {sw:5.0f} wrong = {sw/(sw+W):4.0%} of ALL error, "
          f"overall {(sw+W)/total_after:5.2%}")

2026-08-18 12:45:26.509 | INFO     | stock_predictor.text.coref_eval:load_eval_set:231 - Loaded 270 labelled rows from D:\ML\stock-predictor\data\eval\coref_eval_labelled.parquet (100 span, 170 no-span, 63 errors)
Judge answers: {'yes': 135, 'no': 131, 'unsure': 4} 

channel           err before  err after            95% CI     kept  err recall
coref_span            10.0%       1.6%   [ 0.3%, 8.6%]    62.0%       90.0%
coref_nospan          31.2%       6.8%   [ 3.0%,15.1%]    42.9%       90.6%

WHOLE TARGET-SENTENCE SET
  before judge :    613 wrong of  14605 = 4.20%
  after  judge :     59 wrong of  13068 = 0.45%   (keeps 89.5% of sentences)

  remaining measured error:
    coref_span        24  (41.0%)
    coref_nospan      35  (59.0%)

  surface is UNMEASURED: 11,052 rows = 85% of what survives
    if surface error = 0.5% ->    55 wrong =  48% of ALL error, overall 0.88%
    if surface error = 1.0% ->   111 wrong =  65% of ALL error, overall 1.30%
    if surface error = 2.0% ->   22

The judge clears both pre-registered bars, and error recall is now balanced across the two
populations at 90.0% and 90.6%. Over the whole target set, referent error goes from 4.20% to 0.45%
while keeping 89.5% of sentences.

Attribution matters here. Whole-set error went from 5.11% to 4.20% from the relabelling alone,
before the judge does anything, and the remaining 4.20% to 0.45% is the judge. Quoting 0.45%
against the old 5.11% would credit the model with work the convention did.

The finding that now dominates the error budget is one this branch never measured. `surface`
sentences, where the company is named literally, are 85% of everything that survives and have never
been audited. At a surface error rate of 0.5% they would be 48% of all remaining error, and at 1%
they would be 65%. The coreference channels are clean enough now that they are no longer plausibly
the main source, and the unmeasured one almost certainly is. Section 2 already found one concrete
surface defect, the `Tesla-SpaceX` boundary gap from 2.0 section 2, so treating surface as exact by
construction is an assumption rather than a measurement. The next labelling effort belongs there.

The failures that remain are all no-span and they share a shape. Each is a sentence with no
self-contained subject sitting in a Tesla-saturated context, and the article is Tesla-dominated by
construction, since that is why it was retrieved in the first place. More context would make these
worse rather than better. The fix, if we want one, is to frame the no-span question by predication,
asking whether the target is the subject, rather than topically.

### 10. Conventions and limits

Two labelling conventions changed on 2026-08-18. We applied both to the whole eval set rather than
only to the rows a model happened to accept, because relabelling where a model agrees scores the
labels against the thing under test.

Four categories moved from error to target: the target's own products such as FSD and the Semi,
joint referents that include the target such as "both firms", funds and baskets that hold the
target such as YMAG and BOTT, and generic or third-party statements inside a target article.
Inverse instruments such as TSLQ stay as errors, and that carve-out matters, because an inverse
ETF's sentiment is sign-flipped, so accepting it inverts the signal rather than diluting it. It is
the one case where "reads as a Tesla reference" and "carries Tesla's sentiment" point in opposite
directions.

Together the two changes took the eval set from 85 errors to 63, and the no-span baseline from
56.5% to 68.8%. Nothing about the pipeline changed, only what a label means.

The limits that travel with the good numbers:

- **Ticker-agnostic code is not ticker-uniform accuracy.** The judge works by knowing what Lightship
  and BYD are, that knowledge decays for obscure targets, and nothing detects when it has decayed.
- **All 270 rows are fastcoref's picks.** The judge has never been scored on a sentence fastcoref
  never tagged, and no amount of labelling inside the existing frame fixes that.
- **21 of 270 rows were relabelled** by the second convention change, so the no-span sample now
  carries 53 errors and 6.8% rests on a thinner base than its interval suggests.
- **Recall is measured nowhere here.** Everything above audits what the pipeline claimed and never
  what it missed.

The output is `resolved_by_coref`, `mention_char_start`, `mention_char_end`, and a per-sentence
`judge_accepted` gate. Notebook 2.2 decides what number a surviving sentence carries, and 2.3
section 7 keeps the three provenance channels separable.